# Lesson 7 - Advanced Project

## Scene
Your robot is entering a harder obstacle-course challenge.

This time it should do more than follow a fixed script.
It should sense, decide, recover, and keep going.

This lesson uses the same libraries as Lesson 6, but the code structure should be stronger and more autonomous.

## Project Goal
Use these same libraries:
- `student_robot_moves`
- `eyes_lib`
- `tts_lib`
- `buzzer_lib`
- `camera_lib`
- `sonar_lib` / `ultrasonic_lib`

Build a robot that can make repeated decisions during an obstacle course.

## Advanced Coding Goals
Use:
- variables
- functions that return values
- dictionary lookups
- `for` loops
- `while` loops
- `if / elif / else`
- a class that groups robot state and behaviour
- recovery logic when navigation goes wrong

## Setup
Run this first.

In [ ]:
from lesson_header import *
show_advanced_project_status()

## Libraries In This Lesson
These are the same project libraries from Lesson 6.

In [ ]:
from lesson_header import *

print('student_robot_moves:', getattr(moves, '__file__', None))
print('eyes ready:', eyes is not None)
print('camera ready:', cam is not None)
print('tts ready:', tts is not None)
print('buzzer ready:', bz is not None)
print('sonar ready:', sonar is not None)

## Dictionary Lookup Example
Use dictionaries to map states to colours, words, or actions.

In [ ]:
from lesson_header import *

reaction_colours = {
    'clear': (0, 255, 0),
    'warning': (255, 180, 0),
    'blocked': (255, 0, 0),
    'unknown': (120, 120, 255),
}

reaction_words = {
    'clear': 'Path clear',
    'warning': 'Obstacle near',
    'blocked': 'Blocked',
    'unknown': 'No reading',
}

state = 'warning'
print(reaction_colours[state])
print(reaction_words[state])

## Functions That Return Values
Return values help you separate sensing, deciding, and acting.

In [ ]:
from lesson_header import *

def classify_distance(distance_cm):
    if distance_cm is None:
        return 'unknown'
    if distance_cm < 15:
        return 'blocked'
    if distance_cm < 30:
        return 'warning'
    return 'clear'


def choose_action(state):
    action_map = {
        'blocked': 'turn_left',
        'warning': 'move_left',
        'clear': 'forward',
        'unknown': 'turn_left',
    }
    return action_map[state]

sample = sonar.get_distance_cm(filtered=True)
state = classify_distance(sample)
action = choose_action(state)
print('distance =', sample)
print('state =', state)
print('action =', action)

## Recovery Logic
A stronger robot should recover from errors instead of freezing.

In [ ]:
from lesson_header import *

def recovery_move(state):
    if state == 'blocked':
        moves.backward(0.3)
        moves.turn_left(0.4)
        bz.beep(note='C4', duration_s=0.12)
    elif state == 'warning':
        moves.move_left(0.3)
    else:
        moves.forward(0.3)

state = classify_distance(sonar.get_distance_cm(filtered=True))
recovery_move(state)
stop_project_robot()

## Progress Loop
You can still use obstacle progress and eye colours, but now it should fit inside a stronger structure.

In [ ]:
from lesson_header import *

progress_colours = {
    1: (255, 0, 0),
    2: (0, 255, 0),
    3: (0, 0, 255),
}

obstacles_passed = 2
colour = progress_colours[obstacles_passed]
for i in range(obstacles_passed):
    eyes.set_both(*colour)
    time.sleep(0.2)
    eyes.off()
    time.sleep(0.2)

## Object-Oriented Structure
Use a class to store robot state and methods in one place.

In [ ]:
from lesson_header import *

class CourseRobot:
    def __init__(self, moves_lib, eyes_dev, tts_dev, buzzer_dev, camera_dev, sonar_dev):
        self.moves = moves_lib
        self.eyes = eyes_dev
        self.tts = tts_dev
        self.bz = buzzer_dev
        self.cam = camera_dev
        self.sonar = sonar_dev
        self.obstacles_passed = 0
        self.warning_distance_cm = 25
        self.progress_colours = {
            1: (255, 0, 0),
            2: (0, 255, 0),
            3: (0, 0, 255),
        }

    def get_distance_state(self):
        distance_cm = self.sonar.get_distance_cm(filtered=True)
        if distance_cm is None:
            return 'unknown'
        if distance_cm < 15:
            return 'blocked'
        if distance_cm < self.warning_distance_cm:
            return 'warning'
        return 'clear'

    def choose_move(self, state):
        move_map = {
            'blocked': self.recover_from_block,
            'warning': lambda: self.moves.move_left(0.3),
            'clear': lambda: self.moves.forward(0.3),
            'unknown': lambda: self.moves.turn_left(0.2),
        }
        return move_map[state]

    def recover_from_block(self):
        self.moves.backward(0.3)
        self.moves.turn_left(0.4)
        self.bz.beep(note='C4', duration_s=0.12)

    def show_state(self, state):
        colour_map = {
            'clear': (0, 255, 0),
            'warning': (255, 180, 0),
            'blocked': (255, 0, 0),
            'unknown': (120, 120, 255),
        }
        self.eyes.set_both(*colour_map[state])

    def flash_progress(self):
        colour = self.progress_colours.get(self.obstacles_passed, (255, 255, 255))
        for i in range(self.obstacles_passed):
            self.eyes.set_both(*colour)
            time.sleep(0.15)
            self.eyes.off()
            time.sleep(0.15)

robot = CourseRobot(moves, eyes, tts, bz, cam, sonar)
state = robot.get_distance_state()
robot.show_state(state)
robot.choose_move(state)()
stop_project_robot()
eyes.off()

## Advanced While Loop
The robot should make repeated decisions, not just one move.

In [ ]:
from lesson_header import *

attempts = 0
max_attempts = 3

while attempts < max_attempts:
    state = classify_distance(sonar.get_distance_cm(filtered=True))
    recovery_move(state)
    attempts = attempts + 1

stop_project_robot()
print('attempts:', attempts)

## Advanced Project Skeleton
This is a stronger starting point for a near-autonomous obstacle-course project.

In [ ]:
from lesson_header import *

class AutonomousCourseRobot:
    def __init__(self, moves_lib, eyes_dev, tts_dev, buzzer_dev, camera_dev, sonar_dev):
        self.moves = moves_lib
        self.eyes = eyes_dev
        self.tts = tts_dev
        self.bz = buzzer_dev
        self.cam = camera_dev
        self.sonar = sonar_dev
        self.obstacles_passed = 0
        self.max_obstacles = 3
        self.warning_distance_cm = 25
        self.progress_colours = {
            1: (255, 0, 0),
            2: (0, 255, 0),
            3: (0, 0, 255),
        }

    def get_distance_state(self):
        distance_cm = self.sonar.get_distance_cm(filtered=True)
        if distance_cm is None:
            return 'unknown'
        if distance_cm < 15:
            return 'blocked'
        if distance_cm < self.warning_distance_cm:
            return 'warning'
        return 'clear'

    def choose_move(self, state):
        move_map = {
            'blocked': self.recover_from_block,
            'warning': lambda: self.moves.move_left(0.3),
            'clear': lambda: self.moves.forward(0.3),
            'unknown': lambda: self.moves.turn_left(0.2),
        }
        return move_map[state]

    def recover_from_block(self):
        self.moves.backward(0.3)
        self.moves.turn_left(0.4)
        self.bz.beep(note='C4', duration_s=0.12)

    def flash_progress(self):
        colour = self.progress_colours.get(self.obstacles_passed, (255, 255, 255))
        for i in range(self.obstacles_passed):
            self.eyes.set_both(*colour)
            time.sleep(0.15)
            self.eyes.off()
            time.sleep(0.15)

    def run(self):
        self.tts.say('Starting advanced course run')
        while self.obstacles_passed < self.max_obstacles:
            state = self.get_distance_state()
            action = self.choose_move(state)
            action()
            self.obstacles_passed += 1
            self.flash_progress()
        stop_project_robot()
        self.eyes.off()
        self.tts.say('Advanced course complete')

# advanced_robot = AutonomousCourseRobot(moves, eyes, tts, bz, cam, sonar)
# advanced_robot.run()

## Advanced Student Build Area
Students should now build their own stronger version.
Use the same libraries, but organise the logic so the robot can keep making decisions and recover from mistakes.

In [ ]:
from lesson_header import *

# Variables
warning_distance_cm = 25
max_obstacles = 3
obstacles_passed = 0

# Dictionary ideas
# state_colours = {...}
# move_map = {...}

# Return-value functions
# def classify_distance(distance_cm):
#     return ...
#
# def choose_action(state):
#     return ...

# Class idea
# class MyCourseRobot:
#     def __init__(self, ...):
#         ...
#     def run(self):
#         ...

stop_project_robot()

## Final Reminder
A strong advanced project should feel closer to autonomy:
- sense repeatedly
- decide repeatedly
- recover when needed
- stay organised and reusable